# ADS v2: episode numbering

Groups each customer's events into episodes, as notebook 06: the lifecycle stream (policy code, or the quote's single linked policy, or the quote), the policy cycle the event falls in (renewal: 90 days before to 30 days after the term start; acquisition: 60 before to 30 after; in-life: within the term), sessions split by gaps, then customer / policy / episode-type sequence numbers, EpisodeKey (SCV key : 4-digit sequence) and the review flags.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("21_EpisodeNumbering", "started")


In [ ]:
/******************************************************************************************************/
/*  Name        : ads_v2.ChillEpisodeTypes                                                              */
/*  Description : The episode step types (seeds/ChillEpisodeTypes.csv, an export of lkp.chillepisodetyp */
/*              : es), by code.                                                                         */
/*              :                                                                                       */
/*              : Reads: seeds/ChillEpisodeTypes.csv (written into this cell by the generator)          */
/******************************************************************************************************/
create or replace table ${ads.target}.ChillEpisodeTypes as
select  *
from    values
        ('Home', 'RENEWALS', 'Home Renewal - Customer phones call center', 'HR2b', 'Spine'),
        ('Home', 'ACQUISITION', 'Home Acquisition - TYR Campaign', 'HA2.E1', 'Chill EFfort'),
        ('Home', 'RENEWALS', 'Home Renewal - No terms from the insurer, so no offer is made', 'HR0.F3', 'Friction'),
        ('Home', 'RENEWALS', 'Home Renewal - Customer logs in to portal', 'HR2a', 'Spine'),
        ('Home', 'RENEWALS', 'Home Renewal - Renewal held letter issued', 'HR0.F2', 'Friction'),
        ('Home', 'ACQUISITION', 'Home Acquisition - No price returned, told to call us', 'HA3.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Home Acquisition - Customer gets a price', 'HA3', 'Spine'),
        ('Home', 'ACQUISITION', 'Home Acquisition - Outbound campaigns to quotes that never got a price', 'HA1.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Home Renewal - CRM bounces', 'HR3.B1.1.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Renewal window opens at T-40', 'HR0', 'Spine'),
        ('Home', 'RENEWALS', 'Customer logs in to portal', 'HR2', 'Spine'),
        ('Home', 'RENEWALS', 'Payment taken', 'HR4', 'Spine'),
        ('Home', 'RENEWALS', 'Renewal processed-  policy continued', 'HR5', 'Spine'),
        ('Home', 'RENEWALS', 'Renewal documents dispatched', 'HR6', 'Spine'),
        ('Home', 'ACQUISITION', 'Quote initiated ONLINE', 'HA1', 'Spine'),
        ('Home', 'ACQUISITION', 'Straight into the CALL CENTRE', 'HA0', 'Spine'),
        ('Home', 'ACQUISITION', 'Quote progressed', 'HA2', 'Spine'),
        ('Home', 'ACQUISITION', 'Payment successful', 'HA5', 'Spine'),
        ('Home', 'ACQUISITION', 'Documents requested', 'HA6', 'Spine'),
        ('Home', 'ACQUISITION', 'Policy documents dispatched', 'HA7', 'Spine'),
        ('Home', 'MTA', 'One type can start online-  four cannot', 'HM0', 'Spine'),
        ('Home', 'MTA', 'Customer contacts Chill with a change request', 'HM1', 'Spine'),
        ('Home', 'MTA', 'Agent quotes the new premium and fee', 'HM2', 'Spine'),
        ('Home', 'MTA', 'Customer accepts', 'HM3', 'Spine'),
        ('Home', 'MTA', 'Money due on the change', 'HM4', 'Spine'),
        ('Home', 'MTA', 'MTA processed', 'HM5', 'Spine'),
        ('Home', 'MTA', 'Policy documents reissued', 'HM6', 'Spine'),
        ('Home', 'CANCELLATION', 'Where this journey starts', 'HC0', 'Spine'),
        ('Home', 'CANCELLATION', 'Document chase-  reminder', 'HC2', 'Spine'),
        ('Home', 'CANCELLATION', 'Document chase-  escalation', 'HC3', 'Spine'),
        ('Home', 'CANCELLATION', 'Final notice', 'HC4', 'Spine'),
        ('Home', 'CANCELLATION', 'Cancellation processed', 'HC5', 'Spine'),
        ('Home', 'CANCELLATION', 'Cancellation confirmation sent', 'HC6', 'Spine'),
        ('Home', 'CANCELLATION', 'Refund processed', 'HC7', 'Spine'),
        ('Home', 'DOC REQUESTS', 'No online route', 'HD0', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Customer rings and asks for a document', 'HD1', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Agent locates and sends the document', 'HD2', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Customer receives it', 'HD3', 'Spine'),
        ('Home', 'RENEWALS', 'Renewal invitation sent by EMAIL', 'HR1a', 'Spine'),
        ('Home', 'RENEWALS', 'Renewal invitation sent by POST', 'HR1b', 'Spine'),
        ('Home', 'RENEWALS', 'Customer renews ONLINE', 'HR3a', 'Spine'),
        ('Home', 'RENEWALS', 'Customer renews on an INBOUND call', 'HR3b', 'Spine'),
        ('Home', 'RENEWALS', 'Pay in full', 'HR4a', 'Spine'),
        ('Home', 'RENEWALS', 'Monthly rollover', 'HR4b', 'Spine'),
        ('Home', 'RENEWALS', 'New monthly agreement', 'HR4c', 'Spine'),
        ('Home', 'ACQUISITION', 'Entry channel', 'HA0.C', 'Spine'),
        ('Home', 'ACQUISITION', 'Service retrieve', 'HA1.E1', 'Chill Effort'),
        ('Home', 'MTA', 'Paid now', 'HM4a', 'Spine'),
        ('Home', 'MTA', 'Added to the monthly schedule', 'HM4b', 'Spine'),
        ('Home', 'MTA', 'No money moves', 'HM4c', 'Spine'),
        ('Home', 'CANCELLATION', 'Customer asks to cancel', 'HC1a', 'Spine'),
        ('Home', 'CANCELLATION', 'Trigger detected', 'HC1b', 'Spine'),
        ('Home', 'CLAIMS', 'What this spine is for', 'HCL0', 'Spine'),
        ('Home', 'CLAIMS', 'Handed to the insurer or the provider', 'HCL2', 'Spine'),
        ('Home', 'CLAIMS', 'Claim runs-  invisible to Chill', 'HCL3', 'Spine'),
        ('Home', 'CLAIMS', 'Policy continues with a claim on it', 'HCL4', 'Spine'),
        ('Home', 'ACQUISITION', 'Outbound sales callback', 'HA1.E2', 'Chill Effort'),
        ('Home', 'ACQUISITION', 'Quote retrieved and iterated', 'HA3.B1.1', 'Branch'),
        ('Home', 'ACQUISITION', 'Purchase', 'HA3.B1.2', 'Branch'),
        ('Home', 'ACQUISITION', 'Abandoned or expired', 'HA3.B1.O1', 'Outcome'),
        ('Home', 'ACQUISITION', 'Customer buys ONLINE', 'HA4a', 'Spine'),
        ('Home', 'ACQUISITION', 'Agent quotes and converts on the CALL', 'HA4b', 'Spine'),
        ('Home', 'ACQUISITION', 'Not proceeded with', 'HA4b.O1', 'Outcome'),
        ('Home', 'ACQUISITION', 'Terms of business sent', 'HA5.1', 'Spine'),
        ('Home', 'ACQUISITION', 'Payment and loan setup', 'HA5.B1', 'Branch'),
        ('Home', 'ACQUISITION', 'Loan or Direct Debit initiated with the finance provider', 'HA5.B1.1', 'Branch'),
        ('Home', 'RENEWALS', 'Renewal offer presented at T-30', 'HR0.1', 'Spine'),
        ('Home', 'ACQUISITION', 'Finance application link sent', 'HA5.B1.2', 'Branch'),
        ('Home', 'ACQUISITION', 'Customer completes the application and DD setup', 'HA5.B1.3', 'Branch'),
        ('Home', 'MTA', 'Customer looks for the change online', 'HM0.1', 'Spine'),
        ('Home', 'MTA', 'Property or contents change', 'HM.V1', 'Spine'),
        ('Home', 'MTA', 'Address change', 'HM.V2', 'Spine'),
        ('Home', 'MTA', 'Occupancy or tenancy change', 'HM.V3', 'Spine'),
        ('Home', 'MTA', 'Security or renovation change', 'HM.V4', 'Spine'),
        ('Home', 'MTA', 'Named insured add or remove', 'HM.V5', 'Spine'),
        ('Home', 'MTA', 'Payment schedule change', 'HM.V6', 'Spine'),
        ('Home', 'MTA', 'The type list needs confirming', 'HM.V0', 'Spine'),
        ('Home', 'CANCELLATION', 'Reinstated-  cancellation reversed', 'HC.O1', 'Outcome'),
        ('Home', 'CLAIMS', 'Home claim reported', 'HCL1a', 'Spine'),
        ('Home', 'CLAIMS', 'Escape of water or emergency assistance', 'HCL1b', 'Spine'),
        ('Home', 'CLAIMS', 'Storm-  flood or weather claim', 'HCL1c', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Certificate or proof of cover', 'HD.V1', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Policy schedule or wording', 'HD.V2', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Claims history evidence', 'HD.V3', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Named insured evidence', 'HD.V4', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Letter of indemnity', 'HD.V5', 'Spine'),
        ('Home', 'DOC REQUESTS', 'Direct Debit schedule or payment evidence', 'HD.V6', 'Spine'),
        ('Home', 'IN-LIFE', 'Additional or return premium', 'HAP.1', 'Spine'),
        ('Home', 'ACQUISITION', 'Chase-  inconsistency resolution', 'HA5.B1.4', 'Branch'),
        ('Home', 'ACQUISITION', 'Payment confirmed on the loan path', 'HA5.B1.5', 'Branch'),
        ('Home', 'RENEWALS', 'Declared claims not recorded', 'HR0.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Underinsurance programme', 'HR1.E2', 'Chill Effort'),
        ('Home', 'RENEWALS', 'Customer does not accept the price', 'HR2.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Portal detail updates do not persist', 'HR2.F2', 'Friction'),
        ('Home', 'RENEWALS', 'Online re-quote for another price', 'HR2.B1', 'Branch'),
        ('Home', 'RENEWALS', 'Renewal campaigns: CRM', 'HR1.E1', 'Chill Effort'),
        ('Home', 'RENEWALS', 'Renewal acceptance is not landing', 'HR3.F2', 'Friction'),
        ('Home', 'RENEWALS', 'This year lapsed', 'HR3.B1', 'Branch'),
        ('Home', 'RENEWALS', 'Policy lapsed', 'HR3.O1', 'Outcome'),
        ('Home', 'RENEWALS', 'Payment not resolved-  policy cancelled', 'HR4.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Documents requested', 'HR4.B2', 'Branch'),
        ('Home', 'RENEWALS', 'The suitability letter barely goes out', 'HR6.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Policy fulfilment', 'HR6.B1', 'Branch'),
        ('Home', 'ACQUISITION', 'Switched to pay in full', 'HA5.B1.O1', 'Outcome'),
        ('Home', 'ACQUISITION', 'Cancellation-  loan setup failed', 'HA5.B1.O2', 'Outcome'),
        ('Home', 'ACQUISITION', 'Call wait time-  or fails to make contact', 'HA0.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'We cannot see where customers drop out', 'HA2.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Quote abandoned', 'HA2.F2', 'Friction'),
        ('Home', 'ACQUISITION', 'Web assist: starts online-  then calls to complete', 'HA4.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Payment attempted with friction', 'HA5.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Document intake', 'HA6.B1', 'Branch'),
        ('Home', 'ACQUISITION', 'Number of chases before the customer submits', 'HA6.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Document request issued', 'HA6.B1.1', 'Branch'),
        ('Home', 'ACQUISITION', 'Non-arrival-  and time taken to receive', 'HA7.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Customer submits documents', 'HA6.B1.2', 'Branch'),
        ('Home', 'MTA', 'Call wait time-  or fails to make contact', 'HM1.F1', 'Friction'),
        ('Home', 'MTA', 'The change costs more than the customer expected', 'HM2.F1', 'Friction'),
        ('Home', 'MTA', 'Cancellation-  declined or not proceeded with', 'HM2.O1', 'Outcome'),
        ('Home', 'MTA', 'Document intake', 'HM4.B1', 'Branch'),
        ('Home', 'MTA', 'Payment taken reads nothing', 'HM4.F1', 'Friction'),
        ('Home', 'MTA', 'No confirmation that the change went through', 'HM5.F1', 'Friction'),
        ('Home', 'MTA', 'Non-arrival-  and time taken to receive', 'HM6.F1', 'Friction'),
        ('Home', 'MTA', 'Policy fulfilment', 'HM6.B1', 'Branch'),
        ('Home', 'CANCELLATION', 'The notices may never reach the customer', 'HC3.F1', 'Friction'),
        ('Home', 'CANCELLATION', 'Final notices far outrun cancellations', 'HC4.F1', 'Friction'),
        ('Home', 'CANCELLATION', 'Conditions met-  restored', 'HC4.O1', 'Outcome'),
        ('Home', 'CANCELLATION', 'Refund not received-  or the amount is disputed', 'HC7.F1', 'Friction'),
        ('Home', 'CANCELLATION', 'Upstream feed: Acquisition', 'HC.IN1', 'Spine'),
        ('Home', 'CANCELLATION', 'Upstream feed: Acquisition', 'HC.IN2', 'Spine'),
        ('Home', 'CANCELLATION', 'Upstream feed: Renewal', 'HC.IN3', 'Spine'),
        ('Home', 'CANCELLATION', 'Upstream feed: Renewal', 'HC.IN4', 'Spine'),
        ('Home', 'CANCELLATION', 'Upstream feed: MTA', 'HC.IN5', 'Spine'),
        ('Home', 'CANCELLATION', 'Upstream feed: Arrears', 'HC.IN6', 'Spine'),
        ('Home', 'CANCELLATION', 'Upstream feed: Claims', 'HC.IN7', 'Spine'),
        ('Home', 'ARREARS', 'Who enters this journey', 'HARR.0', 'Spine'),
        ('Home', 'ARREARS', 'Missed payment detected', 'HARR.1', 'Spine'),
        ('Home', 'ARREARS', 'Arrears chase', 'HARR.2', 'Spine'),
        ('Home', 'ARREARS', 'Loan defaults-  arrears not cleared', 'HARR.3', 'Spine'),
        ('Home', 'ARREARS', 'Default chase and final notice', 'HARR.4', 'Spine'),
        ('Home', 'CLAIMS', 'Renewed with the claim priced in', 'HCL.O1', 'Outcome'),
        ('Home', 'CLAIMS', 'Policy ended', 'HCL.O2', 'Outcome'),
        ('Home', 'DOC REQUESTS', 'Call wait time-  or fails to make contact', 'HD1.F1', 'Friction'),
        ('Home', 'DOC REQUESTS', 'The document has to come from the insurer', 'HD2.F1', 'Friction'),
        ('Home', 'DOC REQUESTS', 'Non-arrival-  and the customer rings again', 'HD3.F1', 'Friction'),
        ('Home', 'DOC REQUESTS', 'The same document downloaded again and again', 'HD3.F2', 'Friction'),
        ('Home', 'ACQUISITION', 'Documents validated', 'HA6.B1.3', 'Branch'),
        ('Home', 'ACQUISITION', 'Chase-  first reminder', 'HA6.B1.4', 'Branch'),
        ('Home', 'ACQUISITION', 'Chase-  escalation', 'HA6.B1.5', 'Branch'),
        ('Home', 'ACQUISITION', 'Documents received and validated', 'HA6.B1.6', 'Branch'),
        ('Home', 'ACQUISITION', 'Cancellation-  non-receipt', 'HA6.B1.O1', 'Outcome'),
        ('Home', 'ACQUISITION', 'Policy fulfilment', 'HA7.B1', 'Branch'),
        ('Home', 'ACQUISITION', 'Policy documents dispatched', 'HA7.B1.1', 'Branch'),
        ('Home', 'ACQUISITION', 'Document pack dispatched', 'HA7.B1.2', 'Branch'),
        ('Home', 'ACQUISITION', 'Customer receipt-  implicit', 'HA7.B1.3', 'Branch'),
        ('Home', 'ACQUISITION', 'Non-arrival chase-  inbound', 'HA7.B1.4', 'Branch'),
        ('Home', 'ACQUISITION', 'Replacement requested', 'HA7.B1.5', 'Branch'),
        ('Home', 'RENEWALS', 'Call wait time-  or fails to make contact', 'HR3b.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Loan re-setup with the finance provider', 'HR4c.B1', 'Branch'),
        ('Home', 'MTA', 'Payment schedule change', 'HM4b.B1', 'Branch'),
        ('Home', 'CANCELLATION', 'No retention or save attempt anywhere', 'HC1a.F1', 'Friction'),
        ('Home', 'CANCELLATION', 'The framework says NCT failure', 'HC1b.F1', 'Friction'),
        ('Home', 'ARREARS', 'Arrears cleared', 'HARR.O1', 'Outcome'),
        ('Home', 'ARREARS', 'Pay in full to clear', 'HARR.B1', 'Branch'),
        ('Home', 'ARREARS', 'Payment schedule change', 'HARR.B2', 'Branch'),
        ('Home', 'ARREARS', 'Arrears never cleared', 'HARR.F1', 'Friction'),
        ('Home', 'ARREARS', 'Escalation to forced cancellation', 'HARR.O2', 'Outcome'),
        ('Home', 'CLAIMS', 'Nobody tells the customer anything', 'HCL3.F1', 'Friction'),
        ('Home', 'CLAIMS', 'The renewal price moves because of the claim', 'HCL4.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Re-quote started from the renewal price', 'HR2.B1.1', 'Branch'),
        ('Home', 'RENEWALS', 'New price returned', 'HR2.B1.2', 'Branch'),
        ('Home', 'RENEWALS', 'Customer compares against the renewal offer', 'HR2.B1.3', 'Branch'),
        ('Home', 'RENEWALS', 'Renewal date passes with no renewal', 'HR3.B1.1', 'Branch'),
        ('Home', 'RENEWALS', 'Lapse CRM sequence runs', 'HR3.B1.2', 'Branch'),
        ('Home', 'RENEWALS', 'Document request issued', 'HR4.B2.1', 'Branch'),
        ('Home', 'RENEWALS', 'Customer submits documents', 'HR4.B2.2', 'Branch'),
        ('Home', 'RENEWALS', 'Chase-  first reminder', 'HR4.B2.4', 'Branch'),
        ('Home', 'RENEWALS', 'Chase-  escalation', 'HR4.B2.5', 'Branch'),
        ('Home', 'RENEWALS', 'Documents validated', 'HR4.B2.3', 'Branch'),
        ('Home', 'RENEWALS', 'Documents received and validated', 'HR4.B2.6', 'Branch'),
        ('Home', 'RENEWALS', 'Policy documents dispatched', 'HR6.B1.1', 'Branch'),
        ('Home', 'RENEWALS', 'Document pack dispatched', 'HR6.B1.2', 'Branch'),
        ('Home', 'RENEWALS', 'Customer receipt-  implicit', 'HR6.B1.3', 'Branch'),
        ('Home', 'RENEWALS', 'Non-arrival chase-  inbound', 'HR6.B1.4', 'Branch'),
        ('Home', 'RENEWALS', 'Replacement requested', 'HR6.B1.5', 'Branch'),
        ('Home', 'MTA', 'Only one type in five can be done online', 'HM0.1.F1', 'Friction'),
        ('Home', 'MTA', 'Document request issued', 'HM4.B1.1', 'Branch'),
        ('Home', 'MTA', 'Customer submits documents', 'HM4.B1.2', 'Branch'),
        ('Home', 'MTA', 'Chase-  first reminder', 'HM4.B1.4', 'Branch'),
        ('Home', 'MTA', 'Chase-  escalation', 'HM4.B1.5', 'Branch'),
        ('Home', 'MTA', 'Documents validated', 'HM4.B1.3', 'Branch'),
        ('Home', 'MTA', 'Documents received and validated', 'HM4.B1.6', 'Branch'),
        ('Home', 'MTA', 'Policy documents dispatched', 'HM6.B1.1', 'Branch'),
        ('Home', 'MTA', 'Document pack dispatched', 'HM6.B1.2', 'Branch'),
        ('Home', 'MTA', 'Customer receipt-  implicit', 'HM6.B1.3', 'Branch'),
        ('Home', 'MTA', 'Non-arrival chase-  inbound', 'HM6.B1.4', 'Branch'),
        ('Home', 'MTA', 'Replacement requested', 'HM6.B1.5', 'Branch'),
        ('Home', 'RENEWALS', 'The customer is underinsured at renewal', 'HR1.E2.F1', 'Friction'),
        ('Home', 'RENEWALS', 'The customer shops the renewal at all', 'HR2.B1.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Programme fails and the renewal date is missed', 'HR1.E1.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Chases before the customer submits-  and documents never submitted', 'HR4.B2.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Renews on the renewal offer', 'HR2.B1.O1', 'Outcome'),
        ('Home', 'RENEWALS', 'Renews on a new price', 'HR2.B1.O2', 'Outcome'),
        ('Home', 'RENEWALS', 'Buys elsewhere-  no renewal', 'HR2.B1.O3', 'Outcome'),
        ('Home', 'RENEWALS', 'Renewed inside the window', 'HR3.B1.O1', 'Outcome'),
        ('Home', 'RENEWALS', 'Written as a new policy', 'HR3.B1.O2', 'Outcome'),
        ('Home', 'RENEWALS', 'Lost', 'HR3.B1.O3', 'Outcome'),
        ('Home', 'RENEWALS', 'New loan agreement initiated', 'HR4c.B1.1', 'Branch'),
        ('Home', 'RENEWALS', 'KYC and mandate sent', 'HR4c.B1.2', 'Branch'),
        ('Home', 'RENEWALS', 'Customer signs new mandate', 'HR4c.B1.3', 'Branch'),
        ('Home', 'RENEWALS', 'Chase on the mandate', 'HR4c.B1.4', 'Branch'),
        ('Home', 'RENEWALS', 'Payment confirmed-  new agreement live', 'HR4c.B1.5', 'Branch'),
        ('Home', 'RENEWALS', 'Cancellation-  non-receipt', 'HR4.B2.O1', 'Outcome'),
        ('Home', 'ACQUISITION', 'Mandate not signed-  or the loan is not set up', 'HA5.B1.F1', 'Friction'),
        ('Home', 'ACQUISITION', 'Documents never submitted-  policy at risk of cancellation', 'HA6.B1.F1', 'Friction'),
        ('Home', 'MTA', 'Chases before the customer submits-  and documents never submitted', 'HM4.B1.F1', 'Friction'),
        ('Home', 'MTA', 'Payment schedule change requested', 'HM4b.B1.1', 'Branch'),
        ('Home', 'MTA', 'Schedule change processed by the finance provider', 'HM4b.B1.2', 'Branch'),
        ('Home', 'MTA', 'Cancellation-  non-receipt', 'HM4.B1.O1', 'Outcome'),
        ('Home', 'ARREARS', 'Customer calls to settle the loan', 'HARR.B1.1', 'Branch'),
        ('Home', 'ARREARS', 'Payment taken-  loan closed', 'HARR.B1.2', 'Branch'),
        ('Home', 'RENEWALS', 'Mandate not signed-  or the loan is not set up', 'HR4c.B1.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Cancellation-  re-setup failed', 'HR4c.B1.O1', 'Outcome'),
        ('Home', 'MTA', 'Schedule change not processed', 'HM4b.B1.F1', 'Friction'),
        ('Home', 'ARREARS', 'Cannot raise the lump sum', 'HARR.B1.F1', 'Friction'),
        ('Home', 'ARREARS', 'Schedule change not processed', 'HARR.B2.F1', 'Friction'),
        ('Home', 'ARREARS', 'Not proceeded with', 'HARR.B1.O1', 'Outcome'),
        ('Home', 'RENEWALS', 'No price returned-  or the quote is abandoned', 'HR2.B1.2.F1', 'Friction'),
        ('Home', 'RENEWALS', 'New price is worse than the renewal offer', 'HR2.B1.3.F1', 'Friction'),
        ('Home', 'RENEWALS', 'Unreachable on the lapse sequence', 'HR3.B1.2.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Pre-renewal notification, 60 days', 'R0', 'Spine'),
        ('Motor', 'ACQUISITION', 'Motor Acquisition - Documents never submitted, policy at risk. Bites A5a customers hardest', 'A6.B1.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Motor Renewal - CRM bounces', 'R3.B1.1.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Customer phones call cente', 'R2b', 'Spine'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Renewal customer in an acquisition campaign audience', 'R1.E1.F3', 'Friction'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Declared claims not recorded', 'R0.F2', 'Friction'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Customer logs in to portal', 'R2a', 'Spine'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Renewal held letter', 'R0.B1', 'Branch'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Not sent - no terms from Insurer', 'R0.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Motor Renewal - In-flight price improvement: promo code, or the agent reduces the fee', 'R2.E1', 'Chill EFfort'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Alternative offer sent', 'R1.B2', 'Branch'),
        ('Motor', 'RENEWALS', 'Motor Renewal - Customer renews on CHAT', 'R3d', 'Spine'),
        ('Motor', 'ACQUISITION', 'Quote initiated ONLINE', 'A1', 'Spine'),
        ('Motor', 'ACQUISITION', 'Straight into the CALL CENTRE', 'A0', 'Spine'),
        ('Motor', 'ACQUISITION', 'Customer details complete', 'A2', 'Spine'),
        ('Motor', 'ACQUISITION', 'Customer gets a price', 'A3', 'Spine'),
        ('Motor', 'ACQUISITION', 'Documents requested', 'A6', 'Spine'),
        ('Motor', 'ACQUISITION', 'Cert and disc sent', 'A7', 'Spine'),
        ('Motor', 'RENEWALS', 'Customer logs in to portal', 'R2', 'Spine'),
        ('Motor', 'RENEWALS', 'Payment taken', 'R4', 'Spine'),
        ('Motor', 'RENEWALS', 'Renewal processed-  policy continued', 'R5', 'Spine'),
        ('Motor', 'RENEWALS', 'Cert and disc sent', 'R6', 'Spine'),
        ('Motor', 'MTA', 'No online route', 'M0', 'Spine'),
        ('Motor', 'MTA', 'Customer calls Chill with a change request', 'M1', 'Spine'),
        ('Motor', 'MTA', 'Agent quotes the new premium and fee', 'M2', 'Spine'),
        ('Motor', 'MTA', 'Customer accepts', 'M3', 'Spine'),
        ('Motor', 'MTA', 'Money due on the change', 'M4', 'Spine'),
        ('Motor', 'MTA', 'MTA processed', 'M5', 'Spine'),
        ('Motor', 'MTA', 'Cert and disc sent', 'M6', 'Spine'),
        ('Motor', 'CANCELLATION', 'Where this journey starts', 'C0', 'Spine'),
        ('Motor', 'CANCELLATION', 'Document chase-  reminder', 'C2', 'Spine'),
        ('Motor', 'CANCELLATION', 'Document chase-  escalation', 'C3', 'Spine'),
        ('Motor', 'CANCELLATION', 'Final notice', 'C4', 'Spine'),
        ('Motor', 'CANCELLATION', 'Cancellation processed', 'C5', 'Spine'),
        ('Motor', 'CANCELLATION', 'Cancellation confirmation sent', 'C6', 'Spine'),
        ('Motor', 'CANCELLATION', 'Refund processed', 'C7', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'No online route', 'D0', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Customer rings and asks for a document', 'D1', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Agent locates and sends the document', 'D2', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Customer receives it', 'D3', 'Spine'),
        ('Motor', 'ACQUISITION', 'Customer buys ONLINE', 'A4a', 'Spine'),
        ('Motor', 'ACQUISITION', 'Agent quotes and converts on the CALL', 'A4b', 'Spine'),
        ('Motor', 'ACQUISITION', 'Payment attempted ONLINE', 'A5a', 'Spine'),
        ('Motor', 'ACQUISITION', 'Payment attempted on the CALL', 'A5b', 'Spine'),
        ('Motor', 'RENEWALS', 'Renewal invitation sent by EMAIL', 'R1a', 'Spine'),
        ('Motor', 'RENEWALS', 'Renewal invitation sent by POST', 'R1b', 'Spine'),
        ('Motor', 'RENEWALS', 'Customer renews ONLINE', 'R3a', 'Spine'),
        ('Motor', 'RENEWALS', 'Customer renews on an INBOUND call', 'R3b', 'Spine'),
        ('Motor', 'RENEWALS', 'Customer renews on an OUTBOUND call', 'R3c', 'Spine'),
        ('Motor', 'RENEWALS', 'Pay in full', 'R4a', 'Spine'),
        ('Motor', 'RENEWALS', 'Monthly rollover', 'R4b', 'Spine'),
        ('Motor', 'RENEWALS', 'New monthly agreement', 'R4c', 'Spine'),
        ('Motor', 'MTA', 'Paid now', 'M4a', 'Spine'),
        ('Motor', 'MTA', 'Added to the monthly schedule', 'M4b', 'Spine'),
        ('Motor', 'MTA', 'Return premium', 'M4c', 'Spine'),
        ('Motor', 'CANCELLATION', 'Customer asks to cancel', 'C1a', 'Spine'),
        ('Motor', 'CANCELLATION', 'Trigger detected', 'C1b', 'Spine'),
        ('Motor', 'IN-LIFE', 'Risk validation flags a discrepancy', 'IL1', 'Spine'),
        ('Motor', 'IN-LIFE', 'Customer notified of the repricing', 'IL2', 'Spine'),
        ('Motor', 'IN-LIFE', 'Customer response-  the additional premium cannot be declined', 'IL3', 'Spine'),
        ('Motor', 'IN-LIFE', 'Additional premium paid', 'IL4', 'Spine'),
        ('Motor', 'CLAIMS', 'What this spine is for', 'CL0', 'Spine'),
        ('Motor', 'CLAIMS', 'Handed to the insurer or the provider', 'CL2', 'Spine'),
        ('Motor', 'CLAIMS', 'Claim runs-  invisible to Chill', 'CL3', 'Spine'),
        ('Motor', 'CLAIMS', 'Policy continues with a claim on it', 'CL4', 'Spine'),
        ('Motor', 'ACQUISITION', 'Entry channel', 'A0.C', 'Spine'),
        ('Motor', 'ACQUISITION', 'Purchase confirmation sent', 'A5.1', 'Spine'),
        ('Motor', 'MTA', 'Customer looks for the change online', 'M0.1', 'Spine'),
        ('Motor', 'MTA', 'Vehicle change', 'M.V1', 'Spine'),
        ('Motor', 'MTA', 'Named driver add or remove', 'M.V2', 'Spine'),
        ('Motor', 'MTA', 'Licence update', 'M.V3', 'Spine'),
        ('Motor', 'MTA', 'Address change', 'M.V4', 'Spine'),
        ('Motor', 'MTA', 'Personal detail change', 'M.V5', 'Spine'),
        ('Motor', 'MTA', 'Payment schedule change', 'M.V6', 'Spine'),
        ('Motor', 'CLAIMS', 'Motor claim reported', 'CL1a', 'Spine'),
        ('Motor', 'CLAIMS', 'Breakdown assistance requested', 'CL1b', 'Spine'),
        ('Motor', 'CLAIMS', 'Windscreen claim initiated', 'CL1c', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Replacement cert and disc', 'D.V1', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Proof of no claims bonus', 'D.V2', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Letter of indemnity', 'D.V3', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Direct Debit schedule or payment evidence', 'D.V4', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Named driver evidence', 'D.V5', 'Spine'),
        ('Motor', 'DOC REQUESTS', 'Copy policy documents', 'D.V6', 'Spine'),
        ('Motor', 'ACQUISITION', 'Call wait time-  or fails to make contact', 'A0.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Drop off before completing the details', 'A2.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'No price returned-  told to call us', 'A3.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Web assist: starts online-  then calls to complete', 'A4.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Payment failed', 'A5.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Chases before the customer submits-  and documents never submitted', 'A6.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Non-arrival-  and time taken to receive', 'A7.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Renewal campaigns: CRM', 'R1.E1', 'Chill Effort'),
        ('Motor', 'RENEWALS', 'Failed login', 'R2.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Customer does not accept the price', 'R2.F2', 'Friction'),
        ('Motor', 'RENEWALS', 'Portal detail updates do not persist', 'R2.F3', 'Friction'),
        ('Motor', 'RENEWALS', 'Online re-quote (MFQ) for another price', 'R2.B1', 'Branch'),
        ('Motor', 'RENEWALS', 'Change friction: adjustments at renewal', 'R3.F2', 'Friction'),
        ('Motor', 'RENEWALS', 'This year lapsed', 'R3.B1', 'Branch'),
        ('Motor', 'RENEWALS', 'Policy lapsed', 'R3.O1', 'Outcome'),
        ('Motor', 'RENEWALS', 'Payment failed', 'R4.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Documents requested', 'R4.B2', 'Branch'),
        ('Motor', 'RENEWALS', 'Non-arrival-  and time taken to receive', 'R6.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Cert and disc dispatch', 'R6.B1', 'Branch'),
        ('Motor', 'MTA', 'Call wait time-  or fails to make contact', 'M1.F1', 'Friction'),
        ('Motor', 'MTA', 'The change costs more than the customer expected', 'M2.F1', 'Friction'),
        ('Motor', 'MTA', 'Cancellation-  declined or not proceeded with', 'M2.O1', 'Outcome'),
        ('Motor', 'MTA', 'Document intake', 'M3.B1', 'Branch'),
        ('Motor', 'MTA', 'Payment failed', 'M4.F1', 'Friction'),
        ('Motor', 'MTA', 'No confirmation that the MTA was processed', 'M5.F1', 'Friction'),
        ('Motor', 'MTA', 'Non-arrival-  and time taken to receive', 'M6.F1', 'Friction'),
        ('Motor', 'MTA', 'Cert and disc dispatch', 'M6.B1', 'Branch'),
        ('Motor', 'CANCELLATION', 'The notices may never reach the customer', 'C3.F1', 'Friction'),
        ('Motor', 'CANCELLATION', 'Conditions met-  restored', 'C4.O1', 'Outcome'),
        ('Motor', 'CANCELLATION', 'Reinstated-  cancellation reversed', 'C5.O1', 'Outcome'),
        ('Motor', 'CANCELLATION', 'Retention or save attempt', 'C5.E1', 'Chill Effort'),
        ('Motor', 'CANCELLATION', 'Refund not received-  or the amount is disputed', 'C7.F1', 'Friction'),
        ('Motor', 'ARREARS', 'Who enters this journey', 'ARR.0', 'Spine'),
        ('Motor', 'ARREARS', 'Missed payment detected', 'ARR.1', 'Spine'),
        ('Motor', 'ARREARS', 'Arrears chase-  automated', 'ARR.2', 'Spine'),
        ('Motor', 'ARREARS', 'Loan defaults-  arrears not cleared', 'ARR.3', 'Spine'),
        ('Motor', 'ARREARS', 'Default chase-  Chill outbound and final notice', 'ARR.4', 'Spine'),
        ('Motor', 'IN-LIFE', 'Return premium reversed', 'IL.O1', 'Outcome'),
        ('Motor', 'IN-LIFE', 'Cancellation-  additional premium unresolved', 'IL.O2', 'Outcome'),
        ('Motor', 'CLAIMS', 'Renewed with the claim priced in', 'CL.O1', 'Outcome'),
        ('Motor', 'CLAIMS', 'Policy ended', 'CL.O2', 'Outcome'),
        ('Motor', 'DOC REQUESTS', 'Call wait time-  or fails to make contact', 'D1.F1', 'Friction'),
        ('Motor', 'DOC REQUESTS', 'The document cannot be produced by Chill', 'D2.F1', 'Friction'),
        ('Motor', 'DOC REQUESTS', 'Non-arrival-  and the customer rings again', 'D3.F1', 'Friction'),
        ('Motor', 'DOC REQUESTS', 'The same document downloaded again and again', 'D3.F2', 'Friction'),
        ('Motor', 'RENEWALS', 'Call wait time-  or fails to make contact', 'R3b.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Loan setup with the finance provider', 'R4c.B1', 'Branch'),
        ('Motor', 'MTA', 'Payment schedule change', 'M4b.B1', 'Branch'),
        ('Motor', 'CANCELLATION', 'Call wait time-  or fails to make contact', 'C1a.F1', 'Friction'),
        ('Motor', 'ARREARS', 'Arrears cleared', 'ARR.O1', 'Outcome'),
        ('Motor', 'ARREARS', 'Pay in full to clear', 'ARR.B1', 'Branch'),
        ('Motor', 'ARREARS', 'Payment schedule change', 'ARR.B2', 'Branch'),
        ('Motor', 'ARREARS', 'Arrears never cleared', 'ARR.F1', 'Friction'),
        ('Motor', 'ARREARS', 'Escalation to forced cancellation', 'ARR.O2', 'Outcome'),
        ('Motor', 'CLAIMS', 'Nobody tells the customer anything', 'CL3.F1', 'Friction'),
        ('Motor', 'CLAIMS', 'The renewal price moves because of the claim', 'CL4.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Service retrieve', 'A1.E1', 'Chill Effort'),
        ('Motor', 'ACQUISITION', 'This year retrieve (TYR)', 'A2.E1', 'Chill Effort'),
        ('Motor', 'ACQUISITION', 'Quote retrieve and re-quote cycles', 'A3.B1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Quote retrieved or iterated', 'A3.B1.1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Purchase', 'A3.B1.2', 'Branch'),
        ('Motor', 'ACQUISITION', 'Abandoned or expired', 'A3.B1.O1', 'Outcome'),
        ('Motor', 'ACQUISITION', 'Outbound sales callback', 'A3.E1', 'Chill Effort'),
        ('Motor', 'ACQUISITION', 'Not proceeded with', 'A4b.O1', 'Outcome'),
        ('Motor', 'ACQUISITION', 'Payment and loan setup', 'A5.B1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Loan initiated with the finance provider', 'A5.B1.1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Finance application link sent', 'A5.B1.2', 'Branch'),
        ('Motor', 'ACQUISITION', 'Customer completes the application and DD setup', 'A5.B1.3', 'Branch'),
        ('Motor', 'RENEWALS', 'MFQ quote started from the renewal price', 'R2.B1.1', 'Branch'),
        ('Motor', 'RENEWALS', 'Details carried over or amended', 'R2.B1.2', 'Branch'),
        ('Motor', 'RENEWALS', 'New price returned', 'R2.B1.3', 'Branch'),
        ('Motor', 'RENEWALS', 'Customer compares against the renewal offer', 'R2.B1.4', 'Branch'),
        ('Motor', 'RENEWALS', 'Renewal date passes with no renewal', 'R3.B1.1', 'Branch'),
        ('Motor', 'RENEWALS', 'Lapse CRM sequence runs', 'R3.B1.2', 'Branch'),
        ('Motor', 'RENEWALS', 'Online renewal window closes at day 7', 'R3.B1.3', 'Branch'),
        ('Motor', 'RENEWALS', 'Phone renewal window closes at day 30', 'R3.B1.4', 'Branch'),
        ('Motor', 'RENEWALS', 'Document request issued', 'R4.B2.1', 'Branch'),
        ('Motor', 'RENEWALS', 'Customer submits documents', 'R4.B2.2', 'Branch'),
        ('Motor', 'RENEWALS', 'Chase-  first reminder', 'R4.B2.4', 'Branch'),
        ('Motor', 'RENEWALS', 'Chase-  escalation', 'R4.B2.5', 'Branch'),
        ('Motor', 'RENEWALS', 'Documents validated', 'R4.B2.3', 'Branch'),
        ('Motor', 'RENEWALS', 'Documents received and validated', 'R4.B2.6', 'Branch'),
        ('Motor', 'RENEWALS', 'Cert and disc dispatched', 'R6.B1.1', 'Branch'),
        ('Motor', 'RENEWALS', 'Document pack dispatched', 'R6.B1.2', 'Branch'),
        ('Motor', 'RENEWALS', 'Customer receipt-  implicit', 'R6.B1.3', 'Branch'),
        ('Motor', 'RENEWALS', 'Non-arrival chase-  inbound', 'R6.B1.4', 'Branch'),
        ('Motor', 'RENEWALS', 'Replacement requested', 'R6.B1.5', 'Branch'),
        ('Motor', 'MTA', 'They wanted to do it online and cannot', 'M0.1.F1', 'Friction'),
        ('Motor', 'MTA', 'Document request issued', 'M3.B1.1', 'Branch'),
        ('Motor', 'MTA', 'Customer submits documents', 'M3.B1.2', 'Branch'),
        ('Motor', 'MTA', 'Chase-  first reminder', 'M3.B1.4', 'Branch'),
        ('Motor', 'MTA', 'Chase-  escalation', 'M3.B1.5', 'Branch'),
        ('Motor', 'MTA', 'Documents validated', 'M3.B1.3', 'Branch'),
        ('Motor', 'MTA', 'Documents received and validated', 'M3.B1.6', 'Branch'),
        ('Motor', 'MTA', 'Cert and disc dispatched', 'M6.B1.1', 'Branch'),
        ('Motor', 'MTA', 'Document pack dispatched', 'M6.B1.2', 'Branch'),
        ('Motor', 'MTA', 'Customer receipt-  implicit', 'M6.B1.3', 'Branch'),
        ('Motor', 'MTA', 'Non-arrival chase-  inbound', 'M6.B1.4', 'Branch'),
        ('Motor', 'MTA', 'Replacement requested', 'M6.B1.5', 'Branch'),
        ('Motor', 'CANCELLATION', 'Arrears escalation', 'C1b.IN1', 'Spine'),
        ('Motor', 'ACQUISITION', 'Chase-  inconsistency resolution', 'A5.B1.4', 'Branch'),
        ('Motor', 'ACQUISITION', 'Payment confirmed on the loan path', 'A5.B1.5', 'Branch'),
        ('Motor', 'ACQUISITION', 'Retrieve link does not connect-  or the price has changed', 'A2.E1.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Number of cycles through A1 to A3', 'A3.B1.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Mandate not signed-  or the loan is not set up', 'A5.B1.F1', 'Friction'),
        ('Motor', 'ACQUISITION', 'Switched to pay in full', 'A5.B1.O1', 'Outcome'),
        ('Motor', 'ACQUISITION', 'Cancellation-  loan setup failed', 'A5.B1.O2', 'Outcome'),
        ('Motor', 'ACQUISITION', 'Switched to pay in full-  or cancellation', 'A5.B1.PC.O1', 'Outcome'),
        ('Motor', 'ACQUISITION', 'Loan initiated-  integrated at the quote stage', 'A5.B1.PC1', 'Branch'),
        ('Motor', 'RENEWALS', 'Complaints about the renewal programme', 'R1.E1.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Programme fails and the renewal date is missed', 'R1.E1.F2', 'Friction'),
        ('Motor', 'RENEWALS', 'The customer shops the renewal at all', 'R2.B1.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Renews on the renewal offer', 'R2.B1.O1', 'Outcome'),
        ('Motor', 'RENEWALS', 'Renews on a new MFQ price', 'R2.B1.O2', 'Outcome'),
        ('Motor', 'RENEWALS', 'Buys elsewhere-  no renewal', 'R2.B1.O3', 'Outcome'),
        ('Motor', 'RENEWALS', 'Renewed inside the window', 'R3.B1.O1', 'Outcome'),
        ('Motor', 'RENEWALS', 'Written as a new policy', 'R3.B1.O2', 'Outcome'),
        ('Motor', 'RENEWALS', 'Lost', 'R3.B1.O3', 'Outcome'),
        ('Motor', 'RENEWALS', 'New loan agreement initiated', 'R4c.B1.1', 'Branch'),
        ('Motor', 'RENEWALS', 'KYC and mandate sent', 'R4c.B1.2', 'Branch'),
        ('Motor', 'RENEWALS', 'Customer signs new mandate', 'R4c.B1.3', 'Branch'),
        ('Motor', 'RENEWALS', 'Joint chase-  Chill and the finance provider', 'R4c.B1.4', 'Branch'),
        ('Motor', 'RENEWALS', 'Payment confirmed-  new agreement live', 'R4c.B1.5', 'Branch'),
        ('Motor', 'RENEWALS', 'Chases before the customer submits-  and documents never submitted', 'R4.B2.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Cancellation-  non-receipt', 'R4.B2.O1', 'Outcome'),
        ('Motor', 'MTA', 'Chases before the customer submits-  and documents never submitted', 'M3.B1.F1', 'Friction'),
        ('Motor', 'MTA', 'Payment schedule change requested', 'M4b.B1.1', 'Branch'),
        ('Motor', 'MTA', 'Schedule change processed by the finance provider', 'M4b.B1.2', 'Branch'),
        ('Motor', 'MTA', 'Cancellation-  non-receipt', 'M3.B1.O1', 'Outcome'),
        ('Motor', 'ARREARS', 'Customer calls to settle the loan', 'ARR.B1.1', 'Branch'),
        ('Motor', 'ARREARS', 'Payment taken-  loan closed', 'ARR.B1.2', 'Branch'),
        ('Motor', 'ACQUISITION', 'Finance application link sent by Premium Credit', 'A5.B1.PC2', 'Branch'),
        ('Motor', 'ACQUISITION', 'Customer completes the self service application', 'A5.B1.PC3', 'Branch'),
        ('Motor', 'ACQUISITION', 'Document chase-  provider and Chill', 'A5.B1.PC4', 'Branch'),
        ('Motor', 'ACQUISITION', 'Payment confirmed on the loan path', 'A5.B1.PC5', 'Branch'),
        ('Motor', 'ACQUISITION', 'Document intake', 'A6.B1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Document request issued', 'A6.B1.1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Customer submits documents', 'A6.B1.2', 'Branch'),
        ('Motor', 'ACQUISITION', 'Documents validated', 'A6.B1.3', 'Branch'),
        ('Motor', 'ACQUISITION', 'Chase-  first reminder', 'A6.B1.4', 'Branch'),
        ('Motor', 'ACQUISITION', 'Chase-  escalation', 'A6.B1.5', 'Branch'),
        ('Motor', 'ACQUISITION', 'Documents received and validated', 'A6.B1.6', 'Branch'),
        ('Motor', 'ACQUISITION', 'Cancellation-  non-receipt', 'A6.B1.O1', 'Outcome'),
        ('Motor', 'ACQUISITION', 'Cert and disc dispatch', 'A7.B1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Cert and disc dispatched', 'A7.B1.1', 'Branch'),
        ('Motor', 'ACQUISITION', 'Document pack dispatched', 'A7.B1.2', 'Branch'),
        ('Motor', 'ACQUISITION', 'Customer receipt-  implicit', 'A7.B1.3', 'Branch'),
        ('Motor', 'ACQUISITION', 'Non-arrival chase-  inbound', 'A7.B1.4', 'Branch'),
        ('Motor', 'ACQUISITION', 'Replacement requested', 'A7.B1.5', 'Branch'),
        ('Motor', 'RENEWALS', 'Mandate not signed-  or the loan is not set up', 'R4c.B1.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Cancellation-  re-setup failed', 'R4c.B1.O1', 'Outcome'),
        ('Motor', 'MTA', 'Schedule change not processed', 'M4b.B1.F1', 'Friction'),
        ('Motor', 'ARREARS', 'Cannot raise the lump sum', 'ARR.B1.F1', 'Friction'),
        ('Motor', 'ARREARS', 'Schedule change not processed', 'ARR.B2.F1', 'Friction'),
        ('Motor', 'ARREARS', 'Not proceeded with', 'ARR.B1.O1', 'Outcome'),
        ('Motor', 'RENEWALS', 'No price returned-  or the quote is abandoned', 'R2.B1.3.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'New price is worse than the renewal offer', 'R2.B1.4.F1', 'Friction'),
        ('Motor', 'RENEWALS', 'Unreachable on the lapse sequence', 'R3.B1.2.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'Travel Acquisition - Travel Quote Completed', 'TA1p', 'Spine'),
        ('Travel', 'ACQUISITION', 'Who owns what on travel', 'TA0', 'Spine'),
        ('Travel', 'ACQUISITION', 'Travel quote initiated', 'TA1', 'Spine'),
        ('Travel', 'ACQUISITION', 'Pricing presented', 'TA2', 'Spine'),
        ('Travel', 'ACQUISITION', 'Mapfre agent sells over the phone', 'TA4', 'Spine'),
        ('Travel', 'ACQUISITION', 'Purchase confirmed by next-day CSV', 'TA5', 'Spine'),
        ('Travel', 'RENEWALS', 'Pre-renewal notification', 'TR1', 'Spine'),
        ('Travel', 'RENEWALS', 'Customer logs in to portal', 'TR2', 'Spine'),
        ('Travel', 'RENEWALS', 'Renewal processed', 'TR4', 'Spine'),
        ('Travel', 'RENEWALS', 'Lapsed-  the rollover window closed', 'TR5', 'Spine'),
        ('Travel', 'MTA', 'Mapfre handle every travel change', 'TM0', 'Spine'),
        ('Travel', 'MTA', 'Customer requests a change', 'TM1', 'Spine'),
        ('Travel', 'MTA', 'Mapfre process it as cancel and re-buy', 'TM2', 'Spine'),
        ('Travel', 'MTA', 'Net effect-  a new policy record', 'TM3', 'Spine'),
        ('Travel', 'CANCELLATION', 'Where this journey starts', 'TC0', 'Spine'),
        ('Travel', 'CANCELLATION', 'Cancellation requested', 'TC1', 'Spine'),
        ('Travel', 'CANCELLATION', 'Mapfre process the cancellation', 'TC2', 'Spine'),
        ('Travel', 'RENEWALS', 'Customer reviews and re-purchases ONLINE', 'TR3a', 'Spine'),
        ('Travel', 'RENEWALS', 'Customer renews on the MAPFRE phone line', 'TR3b', 'Spine'),
        ('Travel', 'CLAIMS', 'What this spine is for', 'TCL0', 'Spine'),
        ('Travel', 'CLAIMS', 'Claim initiated', 'TCL1', 'Spine'),
        ('Travel', 'CLAIMS', 'Mapfre own the claim end to end', 'TCL2', 'Spine'),
        ('Travel', 'RENEWALS', 'Policy continues-  re-purchased', 'TR.O1', 'Outcome'),
        ('Travel', 'RENEWALS', 'Gone-  or simply not travelling', 'TR.O2', 'Outcome'),
        ('Travel', 'MTA', 'Cancellation leg', 'TM.O1', 'Outcome'),
        ('Travel', 'MTA', 'Re-purchase leg', 'TM.O2', 'Outcome'),
        ('Travel', 'CANCELLATION', 'Policy ended', 'TC.O1', 'Outcome'),
        ('Travel', 'ACQUISITION', 'The whole online journey reads nothing', 'TA1.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'Single trip and annual multi-trip are not distinguished', 'TA2.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'Abandoned-  or deflected and lost', 'TA3.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'We cannot see the phone sale or its price', 'TA4.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'Chill''s view of travel is a day behind', 'TA5.F1', 'Friction'),
        ('Travel', 'RENEWALS', 'The count is far larger than the book', 'TR1.F1', 'Friction'),
        ('Travel', 'RENEWALS', 'Travel renewal date campaign', 'TR1.E1', 'Chill Effort'),
        ('Travel', 'RENEWALS', 'A travel renewal is a re-purchase', 'TR3.F1', 'Friction'),
        ('Travel', 'RENEWALS', 'Policy fulfilment', 'TR4.B1', 'Branch'),
        ('Travel', 'RENEWALS', 'Lapse on travel may be a season-  not a loss', 'TR5.F1', 'Friction'),
        ('Travel', 'RENEWALS', 'Win-back', 'TR5.B1', 'Branch'),
        ('Travel', 'MTA', 'Chill do not know a change was asked for', 'TM1.F1', 'Friction'),
        ('Travel', 'MTA', 'Travel changes inflate cancellations and new business together', 'TM2.F1', 'Friction'),
        ('Travel', 'CANCELLATION', 'No save attempt-  and no way to make one', 'TC1.F1', 'Friction'),
        ('Travel', 'CANCELLATION', 'No refund step exists', 'TC2.F1', 'Friction'),
        ('Travel', 'CLAIMS', 'Outcome unknown to Chill', 'TCL.O1', 'Outcome'),
        ('Travel', 'CANCELLATION', 'Cancellation leg of a travel change', 'TC1.IN1', 'Spine'),
        ('Travel', 'CLAIMS', 'Three routes in and none of them are Chill''s', 'TCL1.F1', 'Friction'),
        ('Travel', 'CLAIMS', 'A bad travel claim lands on the Chill brand anyway', 'TCL2.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'Quote retrieve and re-quote', 'TA3.B1', 'Branch'),
        ('Travel', 'ACQUISITION', 'Quote retrieved and iterated', 'TA3.B1.1', 'Branch'),
        ('Travel', 'ACQUISITION', 'Purchase', 'TA3.B1.2', 'Branch'),
        ('Travel', 'ACQUISITION', 'Abandoned', 'TA3.B1.O1', 'Outcome'),
        ('Travel', 'ACQUISITION', 'Payment in full ONLINE', 'TA3a', 'Spine'),
        ('Travel', 'ACQUISITION', 'Deflected to the Mapfre contact centre', 'TA3b', 'Spine'),
        ('Travel', 'ACQUISITION', 'Policy fulfilment', 'TA6.B1', 'Branch'),
        ('Travel', 'ACQUISITION', 'Mapfre generate and email the policy documents', 'TA6.B1.1', 'Branch'),
        ('Travel', 'ACQUISITION', 'Customer receives-  silent close', 'TA6.B1.2', 'Branch'),
        ('Travel', 'RENEWALS', 'Policy lapsed-  rollover window closed', 'TR5.B1.1', 'Branch'),
        ('Travel', 'RENEWALS', 'Win-back comms through XP', 'TR5.B1.2', 'Branch'),
        ('Travel', 'RENEWALS', 'Win-back purchase-  as a new policy', 'TR5.B1.3', 'Branch'),
        ('Travel', 'ACQUISITION', 'Cycles before buying-  or giving up', 'TA3.B1.F1', 'Friction'),
        ('Travel', 'ACQUISITION', 'No chase route back to Chill', 'TA6.B1.F1', 'Friction'),
        ('Travel', 'RENEWALS', 'No read on whether win-back works', 'TR5.B1.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Van Renewal - Customer phones call cente', 'VR2b', 'Spine'),
        ('Van', 'RENEWALS', 'Van Renewal - Customer logs in to portal', 'VR2a', 'Spine'),
        ('Van', 'RENEWALS', 'Van Renewal - CRM bounces', 'VR3.B1.1.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Renewal window opens at T-40', 'VR0', 'Spine'),
        ('Van', 'RENEWALS', 'Customer logs in to portal', 'VR2', 'Spine'),
        ('Van', 'RENEWALS', 'Payment taken', 'VR4', 'Spine'),
        ('Van', 'RENEWALS', 'Renewal processed-  policy continued', 'VR5', 'Spine'),
        ('Van', 'RENEWALS', 'Renewal documents dispatched', 'VR6', 'Spine'),
        ('Van', 'ACQUISITION', 'Quote initiated ONLINE', 'VA1', 'Spine'),
        ('Van', 'ACQUISITION', 'Straight into the CALL CENTRE', 'VA0', 'Spine'),
        ('Van', 'ACQUISITION', 'Payment successful', 'VA5', 'Spine'),
        ('Van', 'ACQUISITION', 'Documents requested', 'VA6', 'Spine'),
        ('Van', 'ACQUISITION', 'Cert and disc sent', 'VA7', 'Spine'),
        ('Van', 'MTA', 'No online route', 'VM0', 'Spine'),
        ('Van', 'MTA', 'Customer calls Chill with a change request', 'VM1', 'Spine'),
        ('Van', 'MTA', 'Agent quotes the new premium and fee', 'VM2', 'Spine'),
        ('Van', 'MTA', 'Customer accepts', 'VM3', 'Spine'),
        ('Van', 'MTA', 'Money due on the change', 'VM4', 'Spine'),
        ('Van', 'MTA', 'MTA processed', 'VM5', 'Spine'),
        ('Van', 'MTA', 'Cert and disc sent', 'VM6', 'Spine'),
        ('Van', 'CANCELLATION', 'Where this journey starts', 'VC0', 'Spine'),
        ('Van', 'CANCELLATION', 'Document chase-  reminder', 'VC2', 'Spine'),
        ('Van', 'CANCELLATION', 'Document chase-  escalation', 'VC3', 'Spine'),
        ('Van', 'CANCELLATION', 'Final notice', 'VC4', 'Spine'),
        ('Van', 'CANCELLATION', 'Cancellation processed', 'VC5', 'Spine'),
        ('Van', 'CANCELLATION', 'Cancellation confirmation sent', 'VC6', 'Spine'),
        ('Van', 'CANCELLATION', 'Refund processed', 'VC7', 'Spine'),
        ('Van', 'DOC REQUESTS', 'No online route', 'VD0', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Customer rings and asks for a document', 'VD1', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Agent locates and sends the document', 'VD2', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Customer receives it', 'VD3', 'Spine'),
        ('Van', 'RENEWALS', 'Renewal invitation sent by EMAIL', 'VR1a', 'Spine'),
        ('Van', 'RENEWALS', 'Renewal invitation sent by POST', 'VR1b', 'Spine'),
        ('Van', 'RENEWALS', 'Customer renews ONLINE', 'VR3a', 'Spine'),
        ('Van', 'RENEWALS', 'Customer renews on an INBOUND call', 'VR3b', 'Spine'),
        ('Van', 'RENEWALS', 'New monthly agreement', 'VR4c', 'Spine'),
        ('Van', 'MTA', 'Paid now', 'VM4a', 'Spine'),
        ('Van', 'MTA', 'Added to the monthly schedule', 'VM4b', 'Spine'),
        ('Van', 'MTA', 'Return premium', 'VM4c', 'Spine'),
        ('Van', 'CANCELLATION', 'Customer asks to cancel', 'VC1a', 'Spine'),
        ('Van', 'CANCELLATION', 'Trigger detected', 'VC1b', 'Spine'),
        ('Van', 'CLAIMS', 'What this spine is for', 'VCL0', 'Spine'),
        ('Van', 'CLAIMS', 'Handed to the insurer or the provider', 'VCL2', 'Spine'),
        ('Van', 'CLAIMS', 'Claim runs-  invisible to Chill', 'VCL3', 'Spine'),
        ('Van', 'CLAIMS', 'Policy continues with a claim on it', 'VCL4', 'Spine'),
        ('Van', 'RENEWALS', 'Renewal offer presented at T-30', 'VR0.1', 'Spine'),
        ('Van', 'MTA', 'Customer looks for the change online', 'VM0.1', 'Spine'),
        ('Van', 'MTA', 'The type list needs confirming', 'VM.V0', 'Spine'),
        ('Van', 'MTA', 'Vehicle change', 'VM.V1', 'Spine'),
        ('Van', 'MTA', 'Named driver add or remove', 'VM.V2', 'Spine'),
        ('Van', 'MTA', 'Licence update', 'VM.V3', 'Spine'),
        ('Van', 'MTA', 'Address change', 'VM.V4', 'Spine'),
        ('Van', 'MTA', 'Personal detail change', 'VM.V5', 'Spine'),
        ('Van', 'MTA', 'Payment schedule change', 'VM.V6', 'Spine'),
        ('Van', 'CANCELLATION', 'Reinstated-  cancellation reversed', 'VC.O1', 'Outcome'),
        ('Van', 'CLAIMS', 'Van claim reported', 'VCL1a', 'Spine'),
        ('Van', 'CLAIMS', 'Breakdown assistance requested', 'VCL1b', 'Spine'),
        ('Van', 'CLAIMS', 'Windscreen claim initiated', 'VCL1c', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Replacement cert and disc', 'VD.V1', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Proof of no claims bonus', 'VD.V2', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Named driver evidence', 'VD.V3', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Letter of indemnity', 'VD.V4', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Direct Debit schedule or payment evidence', 'VD.V5', 'Spine'),
        ('Van', 'DOC REQUESTS', 'Copy policy documents', 'VD.V6', 'Spine'),
        ('Van', 'IN-LIFE', 'Additional or return premium', 'VAP.1', 'Spine'),
        ('Van', 'RENEWALS', 'Renewal campaigns: CRM', 'VR1.E1', 'Chill Effort'),
        ('Van', 'RENEWALS', 'Portal detail updates do not persist', 'VR2.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Online re-quote (MFQ) for another price', 'VR2.B1', 'Branch'),
        ('Van', 'RENEWALS', 'Renewal acceptance is not landing', 'VR3.F2', 'Friction'),
        ('Van', 'RENEWALS', 'This year lapsed', 'VR3.B1', 'Branch'),
        ('Van', 'RENEWALS', 'Policy lapsed', 'VR3.O1', 'Outcome'),
        ('Van', 'RENEWALS', 'The whole payment journey is blank', 'VR4.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Documents requested', 'VR4.B2', 'Branch'),
        ('Van', 'RENEWALS', 'Cert or disc not arrived-  customer chases', 'VR6.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Cert and disc dispatch', 'VR6.B1', 'Branch'),
        ('Van', 'ACQUISITION', 'The online journey is missing-  not absent', 'VA1.F1', 'Friction'),
        ('Van', 'ACQUISITION', 'Call wait time-  or fails to make contact', 'VA0.F1', 'Friction'),
        ('Van', 'ACQUISITION', 'The call is recorded and nothing on it is', 'VA4.F1', 'Friction'),
        ('Van', 'ACQUISITION', 'Number of chases before the customer submits', 'VA6.F1', 'Friction'),
        ('Van', 'ACQUISITION', 'Cert or disc not arrived-  customer chases', 'VA7.F1', 'Friction'),
        ('Van', 'MTA', 'Call wait time-  or fails to make contact', 'VM1.F1', 'Friction'),
        ('Van', 'MTA', 'The change costs more than the customer expected', 'VM2.F1', 'Friction'),
        ('Van', 'MTA', 'Cancellation-  declined or not proceeded with', 'VM2.O1', 'Outcome'),
        ('Van', 'MTA', 'Document intake', 'VM3.B1', 'Branch'),
        ('Van', 'MTA', 'Payment failed', 'VM4.F1', 'Friction'),
        ('Van', 'MTA', 'No confirmation that the change went through', 'VM5.F1', 'Friction'),
        ('Van', 'MTA', 'Non-arrival-  and time taken to receive', 'VM6.F1', 'Friction'),
        ('Van', 'MTA', 'Cert and disc dispatch', 'VM6.B1', 'Branch'),
        ('Van', 'CANCELLATION', 'The notices may never reach the customer', 'VC3.F1', 'Friction'),
        ('Van', 'CANCELLATION', 'Notices with nothing either side of them', 'VC4.F1', 'Friction'),
        ('Van', 'CANCELLATION', 'Conditions met-  restored', 'VC4.O1', 'Outcome'),
        ('Van', 'CANCELLATION', 'Refund not received-  or the amount is disputed', 'VC7.F1', 'Friction'),
        ('Van', 'CANCELLATION', 'Upstream feed: Acquisition', 'VC.IN1', 'Spine'),
        ('Van', 'CANCELLATION', 'Upstream feed: Acquisition', 'VC.IN2', 'Spine'),
        ('Van', 'CANCELLATION', 'Upstream feed: Renewal', 'VC.IN3', 'Spine'),
        ('Van', 'CANCELLATION', 'Upstream feed: Renewal', 'VC.IN4', 'Spine'),
        ('Van', 'CANCELLATION', 'Upstream feed: MTA', 'VC.IN5', 'Spine'),
        ('Van', 'CANCELLATION', 'Upstream feed: Arrears', 'VC.IN6', 'Spine'),
        ('Van', 'CANCELLATION', 'Upstream feed: Claims', 'VC.IN7', 'Spine'),
        ('Van', 'ARREARS', 'Who enters this journey', 'VARR.0', 'Spine'),
        ('Van', 'ARREARS', 'Missed payment detected', 'VARR.1', 'Spine'),
        ('Van', 'ARREARS', 'Arrears chase', 'VARR.2', 'Spine'),
        ('Van', 'ARREARS', 'Loan defaults-  arrears not cleared', 'VARR.3', 'Spine'),
        ('Van', 'ARREARS', 'Default chase and final notice', 'VARR.4', 'Spine'),
        ('Van', 'CLAIMS', 'Renewed with the claim priced in', 'VCL.O1', 'Outcome'),
        ('Van', 'CLAIMS', 'Policy ended', 'VCL.O2', 'Outcome'),
        ('Van', 'DOC REQUESTS', 'Call wait time-  or fails to make contact', 'VD1.F1', 'Friction'),
        ('Van', 'DOC REQUESTS', 'The document cannot be produced by Chill', 'VD2.F1', 'Friction'),
        ('Van', 'DOC REQUESTS', 'Non-arrival-  and the customer rings again', 'VD3.F1', 'Friction'),
        ('Van', 'DOC REQUESTS', 'The same document downloaded again and again', 'VD3.F2', 'Friction'),
        ('Van', 'RENEWALS', 'Call wait time-  or fails to make contact', 'VR3b.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Loan re-setup with the finance provider', 'VR4c.B1', 'Branch'),
        ('Van', 'MTA', 'Payment schedule change', 'VM4b.B1', 'Branch'),
        ('Van', 'CANCELLATION', 'No retention or save attempt anywhere', 'VC1a.F1', 'Friction'),
        ('Van', 'CANCELLATION', 'The framework says NCT failure', 'VC1b.F1', 'Friction'),
        ('Van', 'ARREARS', 'Arrears cleared', 'VARR.O1', 'Outcome'),
        ('Van', 'ARREARS', 'Pay in full to clear', 'VARR.B1', 'Branch'),
        ('Van', 'ARREARS', 'Payment schedule change', 'VARR.B2', 'Branch'),
        ('Van', 'ARREARS', 'Arrears never cleared', 'VARR.F1', 'Friction'),
        ('Van', 'ARREARS', 'Escalation to forced cancellation', 'VARR.O2', 'Outcome'),
        ('Van', 'CLAIMS', 'Nobody tells the customer anything', 'VCL3.F1', 'Friction'),
        ('Van', 'CLAIMS', 'The renewal price moves because of the claim', 'VCL4.F1', 'Friction'),
        ('Van', 'RENEWALS', 'MFQ quote started from the renewal price', 'VR2.B1.1', 'Branch'),
        ('Van', 'RENEWALS', 'New price returned', 'VR2.B1.2', 'Branch'),
        ('Van', 'RENEWALS', 'Renewal date passes with no renewal', 'VR3.B1.1', 'Branch'),
        ('Van', 'RENEWALS', 'Lapse CRM sequence runs', 'VR3.B1.2', 'Branch'),
        ('Van', 'RENEWALS', 'Document request issued', 'VR4.B2.1', 'Branch'),
        ('Van', 'RENEWALS', 'Customer submits documents', 'VR4.B2.2', 'Branch'),
        ('Van', 'RENEWALS', 'Chase-  first reminder', 'VR4.B2.4', 'Branch'),
        ('Van', 'RENEWALS', 'Chase-  escalation', 'VR4.B2.5', 'Branch'),
        ('Van', 'RENEWALS', 'Documents validated', 'VR4.B2.3', 'Branch'),
        ('Van', 'RENEWALS', 'Documents received and validated', 'VR4.B2.6', 'Branch'),
        ('Van', 'RENEWALS', 'Cert and disc dispatched', 'VR6.B1.1', 'Branch'),
        ('Van', 'RENEWALS', 'Document pack dispatched', 'VR6.B1.2', 'Branch'),
        ('Van', 'RENEWALS', 'Customer receipt-  implicit', 'VR6.B1.3', 'Branch'),
        ('Van', 'RENEWALS', 'Non-arrival chase-  inbound', 'VR6.B1.4', 'Branch'),
        ('Van', 'RENEWALS', 'Replacement requested', 'VR6.B1.5', 'Branch'),
        ('Van', 'ACQUISITION', 'Entry channel', 'VA0.C', 'Spine'),
        ('Van', 'ACQUISITION', 'Agent quotes and converts on the CALL', 'VA4b', 'Spine'),
        ('Van', 'ACQUISITION', 'Not proceeded with', 'VA4b.O1', 'Outcome'),
        ('Van', 'ACQUISITION', 'Payment and loan setup', 'VA5.B1', 'Branch'),
        ('Van', 'ACQUISITION', 'Loan initiated with the finance provider', 'VA5.B1.1', 'Branch'),
        ('Van', 'ACQUISITION', 'Finance application link sent', 'VA5.B1.2', 'Branch'),
        ('Van', 'ACQUISITION', 'Customer completes the application and DD setup', 'VA5.B1.3', 'Branch'),
        ('Van', 'MTA', 'They wanted to do it online and cannot', 'VM0.1.F1', 'Friction'),
        ('Van', 'MTA', 'Document request issued', 'VM3.B1.1', 'Branch'),
        ('Van', 'MTA', 'Customer submits documents', 'VM3.B1.2', 'Branch'),
        ('Van', 'MTA', 'Chase-  first reminder', 'VM3.B1.4', 'Branch'),
        ('Van', 'MTA', 'Chase-  escalation', 'VM3.B1.5', 'Branch'),
        ('Van', 'MTA', 'Documents validated', 'VM3.B1.3', 'Branch'),
        ('Van', 'MTA', 'Documents received and validated', 'VM3.B1.6', 'Branch'),
        ('Van', 'MTA', 'Cert and disc dispatched', 'VM6.B1.1', 'Branch'),
        ('Van', 'MTA', 'Document pack dispatched', 'VM6.B1.2', 'Branch'),
        ('Van', 'MTA', 'Customer receipt-  implicit', 'VM6.B1.3', 'Branch'),
        ('Van', 'MTA', 'Non-arrival chase-  inbound', 'VM6.B1.4', 'Branch'),
        ('Van', 'MTA', 'Replacement requested', 'VM6.B1.5', 'Branch'),
        ('Van', 'ACQUISITION', 'Chase-  inconsistency resolution', 'VA5.B1.4', 'Branch'),
        ('Van', 'ACQUISITION', 'Payment confirmed on the loan path', 'VA5.B1.5', 'Branch'),
        ('Van', 'ACQUISITION', 'Cancellation-  loan setup failed', 'VA5.B1.O1', 'Outcome'),
        ('Van', 'ACQUISITION', 'Premium Credit loan setup', 'VA5.B1.PC1', 'Branch'),
        ('Van', 'ACQUISITION', 'Document intake', 'VA6.B1', 'Branch'),
        ('Van', 'ACQUISITION', 'Document request issued', 'VA6.B1.1', 'Branch'),
        ('Van', 'ACQUISITION', 'Customer submits documents', 'VA6.B1.2', 'Branch'),
        ('Van', 'ACQUISITION', 'Documents validated', 'VA6.B1.3', 'Branch'),
        ('Van', 'ACQUISITION', 'Chase-  first reminder', 'VA6.B1.4', 'Branch'),
        ('Van', 'ACQUISITION', 'Chase-  escalation', 'VA6.B1.5', 'Branch'),
        ('Van', 'ACQUISITION', 'Documents received and validated', 'VA6.B1.6', 'Branch'),
        ('Van', 'ACQUISITION', 'Cancellation-  non-receipt', 'VA6.B1.O1', 'Outcome'),
        ('Van', 'ACQUISITION', 'Cert and disc dispatch', 'VA7.B1', 'Branch'),
        ('Van', 'RENEWALS', 'The programme is all we can see', 'VR1.E1.F1', 'Friction'),
        ('Van', 'RENEWALS', 'The customer shops the renewal at all', 'VR2.B1.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Chases before the customer submits-  and documents never submitted', 'VR4.B2.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Renews on the renewal offer', 'VR2.B1.O1', 'Outcome'),
        ('Van', 'RENEWALS', 'Renews on a new MFQ price', 'VR2.B1.O2', 'Outcome'),
        ('Van', 'RENEWALS', 'Buys elsewhere-  no renewal', 'VR2.B1.O3', 'Outcome'),
        ('Van', 'RENEWALS', 'Renewed inside the window', 'VR3.B1.O1', 'Outcome'),
        ('Van', 'RENEWALS', 'Written as a new policy', 'VR3.B1.O2', 'Outcome'),
        ('Van', 'RENEWALS', 'Lost', 'VR3.B1.O3', 'Outcome'),
        ('Van', 'RENEWALS', 'New loan agreement initiated', 'VR4c.B1.1', 'Branch'),
        ('Van', 'RENEWALS', 'KYC and mandate sent', 'VR4c.B1.2', 'Branch'),
        ('Van', 'RENEWALS', 'Customer signs new mandate', 'VR4c.B1.3', 'Branch'),
        ('Van', 'RENEWALS', 'Chase on the mandate', 'VR4c.B1.4', 'Branch'),
        ('Van', 'RENEWALS', 'Payment confirmed-  new agreement live', 'VR4c.B1.5', 'Branch'),
        ('Van', 'RENEWALS', 'Cancellation-  non-receipt', 'VR4.B2.O1', 'Outcome'),
        ('Van', 'ACQUISITION', 'Mandate not signed-  or the loan is not set up', 'VA5.B1.F1', 'Friction'),
        ('Van', 'ACQUISITION', 'Documents never submitted-  policy at risk of cancellation', 'VA6.B1.F1', 'Friction'),
        ('Van', 'ACQUISITION', 'Cert and disc dispatched', 'VA7.B1.1', 'Branch'),
        ('Van', 'ACQUISITION', 'Document pack dispatched', 'VA7.B1.2', 'Branch'),
        ('Van', 'MTA', 'Chases before the customer submits-  and documents never submitted', 'VM3.B1.F1', 'Friction'),
        ('Van', 'MTA', 'Payment schedule change requested', 'VM4b.B1.1', 'Branch'),
        ('Van', 'MTA', 'Schedule change processed by the finance provider', 'VM4b.B1.2', 'Branch'),
        ('Van', 'MTA', 'Cancellation-  non-receipt', 'VM3.B1.O1', 'Outcome'),
        ('Van', 'ARREARS', 'Customer calls to settle the loan', 'VARR.B1.1', 'Branch'),
        ('Van', 'ARREARS', 'Payment taken-  loan closed', 'VARR.B1.2', 'Branch'),
        ('Van', 'ACQUISITION', 'Customer receipt-  implicit', 'VA7.B1.3', 'Branch'),
        ('Van', 'RENEWALS', 'Mandate not signed-  or the loan is not set up', 'VR4c.B1.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Cancellation-  re-setup failed', 'VR4c.B1.O1', 'Outcome'),
        ('Van', 'ACQUISITION', 'Non-arrival chase-  inbound', 'VA7.B1.4', 'Branch'),
        ('Van', 'MTA', 'Schedule change not processed', 'VM4b.B1.F1', 'Friction'),
        ('Van', 'ARREARS', 'Cannot raise the lump sum', 'VARR.B1.F1', 'Friction'),
        ('Van', 'ARREARS', 'Schedule change not processed', 'VARR.B2.F1', 'Friction'),
        ('Van', 'ARREARS', 'Not proceeded with', 'VARR.B1.O1', 'Outcome'),
        ('Van', 'ACQUISITION', 'Replacement requested', 'VA7.B1.5', 'Branch'),
        ('Van', 'RENEWALS', 'No price returned-  or the quote is abandoned', 'VR2.B1.2.F1', 'Friction'),
        ('Van', 'RENEWALS', 'Unreachable on the lapse sequence', 'VR3.B1.2.F1', 'Friction')
        as t(Product, Journey, Step, step_id, StepType)

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_episode_source                                                              */
/*  Description : Linked events with their step type, product and journey (EpisodeType) from the seed ( */
/*              : 06 cell 0).                                                                           */
/*              :                                                                                       */
/*              : Reads: ads_v2.ScvEventStream, ads_v2.ChillEpisodeTypes                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_episode_source as
select  e.EventId,
        e.SCV_Customer_Key,
        nullif(upper(trim(e.SourcePolicyReference)), '')                                   as SourcePolicyReference,
        nullif(trim(cast(e.SourceQuoteReference as string)), '')        as SourceQuoteReference,
        e.EventDateTime,
        e.EventDate,
        e.EventTypeId,
        lower(trim(t.StepType))                                         as StepType,
        lower(trim(e.PolicyTypeGroup))                                  as ProductType,
        lower(trim(t.Journey))                                          as EpisodeType,
        case when t.step_id is null then 1 else 0 end                   as NotInSeed
from    ${ads.target}.ScvEventStream e
left join ${ads.target}.ChillEpisodeTypes t on t.step_id = e.EventTypeId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_policy_cycles                                                               */
/*  Description : Policy terms from the sales (06 cell 1): term start = renew date - 1 year, term end = */
/*              :  renew date.                                                                          */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicySale                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_policy_cycles as
select  distinct
        PolicyCode,
        cast(add_months(PolicyRenewDateAdj, -12) as date)               as TermStartDate,
        cast(PolicyRenewDateAdj as date)                                as TermEndDate
from    ${ads.ods}.PolicySale
where   PolicyRenewDateAdj is not null

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_episode_parameters                                                          */
/*  Description : Notebook 06's episode windows and gap (cell 2).                                       */
/*              :                                                                                       */
/*              : Reads:                                                                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_episode_parameters as
select  60                                                              as AcquisitionLeadDays,
        90                                                              as RenewalLeadDays,
        30                                                              as AfterStartGraceDays,
        14                                                              as InLifeGapDays

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_quote_policy_links                                                          */
/*  Description : Quotes that appear on events with a policy code, per customer and product (06 cell 4) */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_episode_source                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_quote_policy_links as
select  SCV_Customer_Key,
        ProductType,
        SourceQuoteReference,
        min(SourcePolicyReference)                                      as LinkedPolicyCode,
        count(distinct SourcePolicyReference)                           as LinkedPolicyCount
from    ${ads.stg}.adsv2_episode_source
where   SourceQuoteReference is not null and SourcePolicyReference is not null and SCV_Customer_Key is not null
group by SCV_Customer_Key, ProductType, SourceQuoteReference

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_episode_prepared                                                            */
/*  Description : The effective policy code (the event's own, else its quote's single linked policy) an */
/*              : d the lifecycle stream key P:<policy>, Q:<quote> or UNKNOWN (06 cells 3 and 5).       */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_episode_source, stg.adsv2_quote_policy_links                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_episode_prepared as
select  r.*,
        case when r.EffectivePolicyCode is not null then concat('P:', r.EffectivePolicyCode)
             when r.SourceQuoteReference is not null then concat('Q:', r.SourceQuoteReference)
             else 'UNKNOWN' end                                         as LifecycleStreamKey
from    (
        select  e.*,
                coalesce(e.SourcePolicyReference,
                         case when q.LinkedPolicyCount = 1 then q.LinkedPolicyCode end) as EffectivePolicyCode,
                case when e.SourcePolicyReference is null and q.LinkedPolicyCount > 1 then 1 else 0 end as AmbiguousQuotePolicy
        from    ${ads.stg}.adsv2_episode_source e
        left join ${ads.stg}.adsv2_quote_policy_links q
                on  e.SCV_Customer_Key = q.SCV_Customer_Key
                and e.ProductType = q.ProductType
                and e.SourceQuoteReference = q.SourceQuoteReference
        ) r

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_episode_cycle                                                               */
/*  Description : The policy term each event falls in (06 cell 6): renewal events 90 days before to 30  */
/*              : days after the term start, acquisition 60 before to 30 after, other journeys within t */
/*              : he term; the nearest term start wins.                                                 */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_episode_prepared, stg.adsv2_episode_parameters, stg.adsv2_policy_cyc */
/*              : les                                                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_episode_cycle as
select  *
from    (
        select  e.*,
                p.TermStartDate                                         as CycleAnchorDate,
                count(p.TermStartDate) over (partition by e.EventId)    as CycleCandidateCount,
                row_number() over (partition by e.EventId
                                   order by abs(datediff(e.EventDate, p.TermStartDate)), p.TermStartDate, p.TermEndDate) as CandidateRank
        from    ${ads.stg}.adsv2_episode_prepared e
        cross join ${ads.stg}.adsv2_episode_parameters cfg
        left join ${ads.stg}.adsv2_policy_cycles p
                on  e.EffectivePolicyCode = p.PolicyCode
                and (   (e.EpisodeType = 'renewals'
                         and e.EventDate >= date_sub(p.TermStartDate, cfg.RenewalLeadDays)
                         and e.EventDate <= date_add(p.TermStartDate, cfg.AfterStartGraceDays))
                     or (e.EpisodeType = 'acquisition'
                         and e.EventDate >= date_sub(p.TermStartDate, cfg.AcquisitionLeadDays)
                         and e.EventDate <= date_add(p.TermStartDate, cfg.AfterStartGraceDays))
                     or (e.EpisodeType not in ('renewals', 'acquisition')
                         and e.EventDate >= p.TermStartDate
                         and e.EventDate < p.TermEndDate))
        ) c
where   CandidateRank = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_episode_sessions                                                            */
/*  Description : Sessions within customer x product x stream x journey x cycle (06 cell 7): a new sess */
/*              : ion at the first event, and, where no cycle was found (or for other journeys), after  */
/*              : a gap longer than the journey's window (acquisition 60, renewal 90, in-life 14 days). */
/*              :  Events without an SCV key or journey are not numbered.                               */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_episode_cycle, stg.adsv2_episode_parameters                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_episode_sessions as
select  b.*,
        sum(b.IsNewEpisode) over (partition by b.SCV_Customer_Key, b.ProductType, b.LifecycleStreamKey,
                                               b.EpisodeType, b.CycleAnchorDate
                                  order by b.EventDateTime, b.EventId
                                  rows between unbounded preceding and current row) as EpisodeSessionNumber
from    (
        select  e.*,
                case when e.PreviousEventDate is null then 1
                     when e.CycleAnchorDate is not null and e.EpisodeType in ('acquisition', 'renewals') then 0
                     when datediff(e.EventDate, e.PreviousEventDate) >
                          case when e.EpisodeType = 'acquisition' then cfg.AcquisitionLeadDays
                               when e.EpisodeType = 'renewals' then cfg.RenewalLeadDays
                               else cfg.InLifeGapDays end then 1
                     else 0 end                                         as IsNewEpisode
        from    (
                select  c.*,
                        lag(c.EventDate) over (partition by c.SCV_Customer_Key, c.ProductType, c.LifecycleStreamKey,
                                                            c.EpisodeType, c.CycleAnchorDate
                                               order by c.EventDateTime, c.EventId) as PreviousEventDate
                from    ${ads.stg}.adsv2_episode_cycle c
                where   c.SCV_Customer_Key is not null and c.EventDate is not null
                and     c.ProductType is not null and c.EpisodeType is not null
                ) e
        cross join ${ads.stg}.adsv2_episode_parameters cfg
        ) b

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_episode_instances                                                           */
/*  Description : One row per episode (06 cell 8): first event, first spine event (the episode starts t */
/*              : here when it has one), sequence numbers per customer, per customer x product x stream */
/*              :  and per episode type, and the review flags.                                          */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_episode_sessions                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_episode_instances as
select  s.*,
        row_number() over (partition by s.SCV_Customer_Key
                           order by s.EpisodeStartDate, s.ProductType, s.LifecycleStreamKey, s.EpisodeType,
                                    s.CycleAnchorDate, s.EpisodeSessionNumber) as CustomerEpisodeSequence,
        row_number() over (partition by s.SCV_Customer_Key, s.ProductType, s.LifecycleStreamKey
                           order by s.EpisodeStartDate, s.EpisodeType, s.CycleAnchorDate,
                                    s.EpisodeSessionNumber)             as PolicyEpisodeSequence,
        row_number() over (partition by s.SCV_Customer_Key, s.ProductType, s.LifecycleStreamKey, s.EpisodeType
                           order by s.EpisodeStartDate, s.CycleAnchorDate, s.EpisodeSessionNumber) as EpisodeTypeOccurrence,
        case when s.FirstSpineDate is null then 1 else 0 end            as MissingSpine,
        case when s.FirstEventDate < s.FirstSpineDate then 1 else 0 end as EventsBeforeSpine,
        case when s.CycleAnchorDate is null then 1 else 0 end           as UsedGapFallback
from    (
        select  i.*, coalesce(i.FirstSpineDate, i.FirstEventDate)       as EpisodeStartDate
        from    (
                select  SCV_Customer_Key, ProductType, LifecycleStreamKey, EpisodeType, CycleAnchorDate,
                        EpisodeSessionNumber,
                        min(EventDate)                                  as FirstEventDate,
                        min(case when StepType = 'spine' then EventDate end) as FirstSpineDate,
                        max(EventDate)                                  as LastEventDate,
                        count(*)                                        as EventCount,
                        max(AmbiguousQuotePolicy)                       as AmbiguousQuotePolicy,
                        max(CycleCandidateCount)                        as MaxCycleCandidateCount
                from    ${ads.stg}.adsv2_episode_sessions
                group by SCV_Customer_Key, ProductType, LifecycleStreamKey, EpisodeType, CycleAnchorDate,
                         EpisodeSessionNumber
                ) i
        ) s

In [ ]:
/******************************************************************************************************/
/*  Name        : ads_v2.EpisodeEventsNumbered                                                          */
/*  Description : Every event with its episode (06 cells 9-10 in one table): EpisodeKey = SCV key : 4-d */
/*              : igit customer episode sequence. NeedsReview = 1 when the event is not numbered (no SC */
/*              : V key, or code not in the seed) or its episode has no spine, events before the spine, */
/*              :  no cycle (gap fallback), an ambiguous quote -> policy link, several cycle candidates */
/*              :  or an UNKNOWN stream. NotNumberedReason says why an event has no episode.            */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_episode_source, stg.adsv2_episode_sessions, stg.adsv2_episode_instan */
/*              : ces                                                                                   */
/******************************************************************************************************/
create or replace table ${ads.target}.EpisodeEventsNumbered as
select  src.EventId,
        src.SCV_Customer_Key,
        src.SourcePolicyReference,
        src.SourceQuoteReference,
        src.EventDateTime,
        src.EventDate,
        src.EventTypeId,
        src.StepType,
        src.ProductType,
        src.EpisodeType,
        e.EffectivePolicyCode,
        e.LifecycleStreamKey,
        e.CycleAnchorDate,
        e.EpisodeSessionNumber,
        i.CustomerEpisodeSequence,
        i.PolicyEpisodeSequence,
        i.EpisodeTypeOccurrence,
        i.EpisodeStartDate,
        i.MissingSpine,
        i.EventsBeforeSpine,
        i.UsedGapFallback,
        i.AmbiguousQuotePolicy,
        i.MaxCycleCandidateCount,
        src.NotInSeed,
        case when src.SCV_Customer_Key is null then 'no SCV key'
             when src.NotInSeed = 1 then 'code not in ChillEpisodeTypes'
             when i.CustomerEpisodeSequence is null then 'not numbered' end as NotNumberedReason,
        case when i.CustomerEpisodeSequence is null then 1
             when i.MissingSpine = 1 or i.EventsBeforeSpine = 1 or i.UsedGapFallback = 1
               or i.AmbiguousQuotePolicy = 1 or i.MaxCycleCandidateCount > 1
               or e.LifecycleStreamKey = 'UNKNOWN' then 1
             else 0 end                                                 as NeedsReview,
        case when i.CustomerEpisodeSequence is not null
             then concat(cast(src.SCV_Customer_Key as string), ':', lpad(cast(i.CustomerEpisodeSequence as string), 4, '0'))
        end                                                             as EpisodeKey,
        '${ads.batch_id}'                                               as AdsBatchId
from    ${ads.stg}.adsv2_episode_source src
left join ${ads.stg}.adsv2_episode_sessions e on e.EventId = src.EventId
left join ${ads.stg}.adsv2_episode_instances i
        on  e.SCV_Customer_Key = i.SCV_Customer_Key
        and e.ProductType = i.ProductType
        and e.LifecycleStreamKey = i.LifecycleStreamKey
        and e.EpisodeType = i.EpisodeType
        and e.CycleAnchorDate <=> i.CycleAnchorDate
        and e.EpisodeSessionNumber = i.EpisodeSessionNumber

In [ ]:
%%pyspark
# numbering summary: Events numbered / not numbered, episodes and review flags by product and journey (printed).
#
# Reads: ads_v2.EpisodeEventsNumbered
spark.sql(f"""
    select  ProductType, EpisodeType, count(*) as Events,
            sum(case when EpisodeKey is null then 1 else 0 end) as NotNumbered,
            count(distinct EpisodeKey) as Episodes,
            sum(NeedsReview) as NeedsReview, sum(coalesce(MissingSpine, 0)) as MissingSpine,
            sum(coalesce(UsedGapFallback, 0)) as GapFallback
    from    {target_schema}.EpisodeEventsNumbered
    group by ProductType, EpisodeType
    order by ProductType, EpisodeType""").show(100, truncate=False)
spark.sql(f"""
    select  NotNumberedReason, count(*) as Events
    from    {target_schema}.EpisodeEventsNumbered
    where   NotNumberedReason is not null
    group by NotNumberedReason""").show(truncate=False)


## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to AdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "ChillEpisodeTypes", "bk": ["step_id"], "sk": null, "nullable": [], "allow": []}, {"table": "EpisodeEventsNumbered", "bk": ["EventId"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "AdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.AdsBuildLog")
if failures:
    ads_run_log("21_EpisodeNumbering", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ads checks failed:\n- " + "\n- ".join(failures))
ads_run_log("21_EpisodeNumbering", "succeeded")


In [ ]:
%%pyspark
ads_run_log("21_EpisodeNumbering", "succeeded")
